# qBraid Quantum Labs: IBM Devices with your own credentials

Authors: Sophy Shin, James Weaver, Brian Ingmanson

Updated by: Harshit Gupta

This tutorial demonstrates how to run quantum circuits using either a **local Aer simulator** (no credentials required) or **IBM Quantum backends** via Qiskit Runtime through qBraid Quantum Lab.

By default, the notebook uses the Aer simulator for quick local testing. To switch to a real IBM backend, set `USE_SIMULATOR = False` in the configuration cell below and provide your IBM Quantum credentials.

### Using a real IBM backend

If you want to use a real IBM Quantum backend, you will need to bring your own credential from [IBM Quantum Platform](https://quantum.ibm.com/).

If you do not already have a user account, get one at the [IBM Quantum login page](https://quantum.ibm.com/login). Your user account is associated with one or more [instances](https://docs.quantum.ibm.com/run/instances) (in the form hub / group / project) that give access to IBM Quantum services. Additionally, a unique token is assigned to each account, allowing for IBM Quantum access from Qiskit. The instructions in this section use our default instance. For instructions to choose a specific instance, see [Connect to an instance](https://docs.quantum.ibm.com/run/instances#connect-instance).

If you are using `Open Plan` you can run your quantum circuits on IBM quantum systems for free (up to 10 minutes quantum time per month). See [IBM Quantum access plans](https://www.ibm.com/quantum/pricing) for details.

In [ ]:
%matplotlib inline

In [ ]:
# Toggle this flag to switch between local simulator and real IBM backend
USE_SIMULATOR = False

## 1. Using Qiskit Runtime

Set up the backend. When `USE_SIMULATOR = True`, we use the local `AerSimulator`. Otherwise, we connect to IBM Quantum via `QiskitRuntimeService` and select the least busy backend.

In [ ]:
if USE_SIMULATOR:
    from qiskit_aer import AerSimulator

    backend = AerSimulator()
    print(f"Using local simulator: {backend.name}")
else:
    from qiskit_ibm_runtime import QiskitRuntimeService

    # Replace with your own credentials, or set USE_SIMULATOR = True to skip
    service = QiskitRuntimeService(
        channel="ibm_cloud",
        instance="<MY_IBM_SERVICE_CRN>",
        token="<MY_IBM_QUANTUM_TOKEN>",
    )
    backend = service.least_busy(simulator=False, operational=True)
    print(f"Using IBM backend: {backend.name}")

### Create a toy circuit

Now, let's create a simple Bell state circuit using Qiskit:

In [ ]:
from qiskit import QuantumCircuit

circ = QuantumCircuit(2)
circ.h(0)
circ.cx(0, 1)
circ.measure_all()

circ.draw()

### Execute using a quantum primitive function

Quantum computers can produce random results, so you'll often want to collect a sample of the outputs by running the circuit many times. Qiskit provides two [primitives](https://docs.quantum.ibm.com/run/primitives-get-started): `Sampler` (measurement counts) and `Estimator` (observable expectation values).

Here we use the `Estimator` to compute the expectation value of the `ZZ` observable on our Bell state. The Estimator requires a circuit **without** terminal measurements, so we create a copy with measurements removed.

In [ ]:
from qiskit.quantum_info import SparsePauliOp

# Estimator needs a circuit without final measurements
estimator_circ = circ.remove_final_measurements(inplace=False)

n_qubits = 2
observable = SparsePauliOp("Z" * n_qubits)

if USE_SIMULATOR:
    from qiskit_aer.primitives import EstimatorV2 as Estimator

    estimator = Estimator()
    job = estimator.run([(estimator_circ, observable)])
else:
    from qiskit_ibm_runtime import EstimatorV2 as Estimator
    from qiskit.transpiler.preset_passmanagers import generate_preset_pass_manager

    estimator = Estimator(backend)
    pm = generate_preset_pass_manager(optimization_level=1, backend=backend)
    isa_circuit = pm.run(estimator_circ)
    isa_observable = observable.apply_layout(isa_circuit.layout)
    job = estimator.run([(isa_circuit, isa_observable)])

### View the result

The Estimator V2 result is accessed via `result[0].data.evs`. For a Bell state measured with `ZZ`, the expected value should be close to `1.0` (both qubits are always in the same state).

In [ ]:
result = job.result()
print(f"Expectation value <ZZ>: {result[0].data.evs}")

## 2. Using qBraid-SDK

In this section we run the same Bell state circuit using the qBraid SDK and visualize the measurement counts.

When `USE_SIMULATOR = True`, we run directly on `AerSimulator` and use `qbraid.visualization` for plotting. When `USE_SIMULATOR = False`, we use `QiskitRuntimeProvider` to submit to a real IBM backend.

First, check the installed qBraid version:

In [ ]:
import qbraid

qbraid.__version__

### Set up the backend

When using the simulator path, we run the circuit directly on `AerSimulator`. For the real backend path, we set up a `QiskitRuntimeProvider` with your IBM credentials and select a device.

In [ ]:
if USE_SIMULATOR:
    from qiskit_aer import AerSimulator

    aer_backend = AerSimulator()
    print(f"Using local simulator: {aer_backend.name}")
else:
    import os

    from qbraid.runtime import QiskitRuntimeProvider

    token = os.getenv("QISKIT_IBM_TOKEN") or ""
    instance = os.getenv("QISKIT_IBM_INSTANCE") or ""

    provider = QiskitRuntimeProvider(token=token, instance=instance)
    print("Available devices:")
    print(provider.get_devices())

### Run the circuit and collect results

In [ ]:
if USE_SIMULATOR:
    aer_job = aer_backend.run(circ, shots=200)
    aer_result = aer_job.result()
    measurement_counts = aer_result.get_counts()
else:
    ibm_device = provider.get_device("ibm_kingston")  # replace with your preferred backend
    qbraid_ibm_job = ibm_device.run(circ, shots=200)
    print(f"Job status: {qbraid_ibm_job.status()}")
    ibm_result = qbraid_ibm_job.result()
    measurement_counts = ibm_result.data.get_counts()

print(f"Measurement counts: {measurement_counts}")

In [ ]:
from qbraid.visualization import plot_histogram, plot_distribution

plot_histogram(measurement_counts)

You can also plot the probability distribution with `plot_distribution`:

In [ ]:
plot_distribution(measurement_counts)

<div class="alert alert-block alert-info">
<b>Copyright Notice:</b> 
    All rights reserved © [2026] qBraid. This notebook is part of the qBraid-Lab-Demo repository.
The qBraid-Lab-Demo is licensed under the Apache License, Version 2.0.
You may obtain a copy of the License at <https://www.apache.org/licenses/LICENSE-2.0>.
Unless required by applicable law or agreed to in writing, software distributed under the License is distributed on an "AS IS" BASIS, WITHOUT WARRANTIES OR CONDITIONS OF ANY KIND, either express or implied.
</div>